# 04 RFM Customer Segmentation

This notebook builds an RFM customer segmentation model using the cleanned product sales dataset.

RFM stands for:
- Recency: how recently a customer purchased
- Frequency: how ofthen a customer purchased
- Monetary: how much revenue a customer contributed

The goal is to identify customer value segments and prepare customer-level outputs for business recommendations and dashborad development.

Scope:
- Build cusomer-level RFM table
- Score customer by recency, frequency, and monetary value
- Assign customer segments
- Summarize segment-level business performence
- Export RFM results

In [1]:
import pandas as pd
import numpy as np

In [2]:
clean_product_sales = pd.read_csv("../data/processed/clean_product_sales.csv")
clean_all_transactions = pd.read_csv("../data/processed/clean_all_transactions.csv")

clean_product_sales["invoice_date"] = pd.to_datetime(clean_product_sales["invoice_date"])
clean_all_transactions["invoice_date"] = pd.to_datetime(clean_all_transactions["invoice_date"])

clean_all_transactions["is_cancelled"] = (
    clean_all_transactions["is_cancelled"]
    .astype(str)
    .str.lower()
    .eq("true")
)

In [3]:
non_product_stockcodes = [
    "M",
    "AMAZONFEE",
    "POST",
    "B",
    "BANK CHARGES",
    "C2",
    "DOT",
    "PADS"
]

clean_all_transactions["is_non_product_transaction"] = (
    clean_all_transactions["stockcode"].isin(non_product_stockcodes) |
    clean_all_transactions["description"].str.upper().str.contains(
        "AMAZON FEE|POSTAGE|MANUAL|ADJUST BAD DEBT|BANK CHARGES|DOTCOM POSTAGE",
        na=False
    )
)

product_transactions = clean_all_transactions[
    (clean_all_transactions["is_non_product_transaction"] == False) &
    (clean_all_transactions["customer_id"].notna())
].copy()

product_transactions["customer_id"] = (
    product_transactions["customer_id"]
    .astype("Int64")
    .astype(str)
)

In [4]:
positive_product_purchases = product_transactions[
    (product_transactions["quantity"] > 0) &
    (product_transactions["is_cancelled"] == False)
].copy()

analysis_date = positive_product_purchases["invoice_date"].max() + pd.Timedelta(days=1)

analysis_date

Timestamp('2011-12-10 12:50:00')

In [5]:
rfm_behavior = (
    positive_product_purchases
    .groupby("customer_id")
    .agg(
        last_purchase_date=("invoice_date", "max"),
        recency=("invoice_date", lambda x: (analysis_date - x.max()).days),
        frequency=("invoice", "nunique")
    )
    .reset_index()
)

rfm_monetary = (
    product_transactions
    .groupby("customer_id")
    .agg(
        monetary_value=("revenue", "sum")
    )
    .reset_index()
)

rfm = rfm_behavior.merge(
    rfm_monetary,
    on="customer_id",
    how="left"
)

rfm.head()

,customer_id,last_purchase_date,recency,frequency,monetary_value
0,12346,2011-01-18 10:01:00,326,12,263.86
1,12347,2011-12-07 15:52:00,2,8,4921.53
2,12348,2011-09-25 13:13:00,75,5,1658.40
3,12349,2011-11-21 09:51:00,19,3,3654.54
4,12350,2011-02-02 16:01:00,310,1,294.40


In [6]:
rfm[['recency', 'frequency', 'monetary_value']].describe()

,recency,frequency,monetary_value
count,5861.000000,5861.000000,5861.000000
mean,200.937895,6.252175,2788.351215
std,209.201911,12.778230,13790.243814
min,1.000000,1.000000,-1343.240000
25%,26.000000,1.000000,329.420000
50%,96.000000,3.000000,840.050000
75%,379.000000,7.000000,2165.760000
max,739.000000,376.000000,577180.220000


Some customers have negative net monetary value because product return or cancellation records may exceed positive product purchase records within the available data window. This can happen when the original purchase occurred before the dataset period, when return records cannot be perfectly matched to original invoices, or when product-level filtering excludes non-product positive charges. These customers are retained and labeled separately as Negative Net Value Customers.

In [7]:
negative_monetary_customers = rfm[rfm["monetary_value"] <= 0].copy()

negative_monetary_summary = pd.DataFrame({
    "metric": [
        "negative_net_value_customers",
        "negative_net_value_customer_share",
        "total_negative_monetary_value",
        "min_monetary_value"
    ],
    "value": [
        len(negative_monetary_customers),
        len(negative_monetary_customers) / len(rfm),
        negative_monetary_customers["monetary_value"].sum(),
        negative_monetary_customers["monetary_value"].min()
    ]
})

negative_monetary_summary

,metric,value
0,negative_net_value_customers,18.000000
1,negative_net_value_customer_share,0.003071
2,total_negative_monetary_value,-1555.560000
3,min_monetary_value,-1343.240000


In [8]:
rfm["recency_score"] = pd.cut(
    rfm["recency"].rank(method="average", pct=True, ascending=False),
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

rfm["frequency_score"] = pd.cut(
    rfm["frequency"].rank(method="average", pct=True, ascending=True),
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

rfm["monetary_score"] = pd.cut(
    rfm["monetary_value"].rank(method="average", pct=True, ascending=True),
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

rfm["rfm_score"] = (
    rfm["recency_score"].astype(str) +
    rfm["frequency_score"].astype(str) +
    rfm["monetary_score"].astype(str)
)

rfm["rfm_total_score"] = (
    rfm["recency_score"] +
    rfm["frequency_score"] +
    rfm["monetary_score"]
)

In [9]:
def assign_rfm_segment(row):
    if row['monetary_value'] <= 0:
        return 'Negative Net Value Customers'
    elif row['recency_score'] >= 4 and row['frequency_score'] >= 4 and row['monetary_score'] >= 4:
        return 'Champions'
    elif row['recency_score'] >= 4 and row['frequency_score'] >= 3:
        return 'Loyal Customers'
    elif row['recency_score'] >= 4 and row['frequency_score'] <= 2:
        return 'New or Recent Customers'
    elif row['recency_score'] <= 2 and row['frequency_score'] >= 4 and row['monetary_score'] >= 4:
        return 'At-Risk High-Value Customers'
    elif row['recency_score'] <= 2 and row['frequency_score'] >= 3:
        return 'At-Risk Loyal Customers'
    elif row['recency_score'] <= 2 and row['monetary_score'] >= 4:
        return 'At-Risk High-Spending Customers'
    elif row['rfm_total_score'] >= 10:
        return 'Valuable Regular Customers'
    elif row['rfm_total_score'] <= 5:
        return 'Low-Value Inactive Customers'
    else:
        return 'Regular Customers'

rfm['customer_segment'] = rfm.apply(assign_rfm_segment, axis=1)

rfm.head()

,customer_id,last_purchase_date,recency,frequency,monetary_value,recency_score,frequency_score,monetary_score,rfm_score,rfm_total_score,customer_segment
0,12346,2011-01-18 10:01:00,326,12,263.86,2,5,1,251,8,At-Risk Loyal Customers
1,12347,2011-12-07 15:52:00,2,8,4921.53,5,4,5,545,14,Champions
2,12348,2011-09-25 13:13:00,75,5,1658.40,3,4,4,344,11,Valuable Regular Customers
3,12349,2011-11-21 09:51:00,19,3,3654.54,5,3,5,535,13,Loyal Customers
4,12350,2011-02-02 16:01:00,310,1,294.40,2,1,2,212,5,Low-Value Inactive Customers


In [10]:
segment_summary = (
    rfm
    .groupby("customer_segment")
    .agg(
        customers=('customer_id', 'count'),
        avg_recency=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        total_monetary_value=('monetary_value', 'sum'),
        avg_monetary_value=('monetary_value', 'mean'),
        avg_rfm_total_score=('rfm_total_score', 'mean')
    )
    .reset_index()
)

segment_summary['customer_share'] = (
    segment_summary['customers'] / segment_summary['customers'].sum()
)

segment_summary['monetary_share'] = (
    segment_summary['total_monetary_value'] / segment_summary['total_monetary_value'].sum()
)

segment_summary = segment_summary.sort_values(by='total_monetary_value', ascending=False)

segment_summary_formatted = segment_summary.copy()

segment_summary_formatted["avg_recency"] = segment_summary_formatted["avg_recency"].round(1)
segment_summary_formatted["avg_frequency"] = segment_summary_formatted["avg_frequency"].round(1)
segment_summary_formatted["total_monetary_value"] = segment_summary_formatted["total_monetary_value"].round(2)
segment_summary_formatted["avg_monetary_value"] = segment_summary_formatted["avg_monetary_value"].round(2)
segment_summary_formatted["avg_rfm_total_score"] = segment_summary_formatted["avg_rfm_total_score"].round(1)
segment_summary_formatted["customer_share"] = segment_summary_formatted["customer_share"].apply(lambda x: f"{x:.2%}")
segment_summary_formatted["monetary_share"] = segment_summary_formatted["monetary_share"].apply(lambda x: f"{x:.2%}")

segment_summary_formatted

,customer_segment,customers,avg_recency,avg_frequency,total_monetary_value,avg_monetary_value,avg_rfm_total_score,customer_share,monetary_share
3,Champions,1247,19.2,17.4,11213228.22,8992.16,13.9,21.28%,68.61%
9,Valuable Regular Customers,570,103.6,8.0,1871440.11,3283.23,11.4,9.73%,11.45%
1,At-Risk High-Value Customers,207,340.9,9.6,844312.27,4078.80,10.4,3.53%,5.17%
5,Loyal Customers,576,24.4,4.0,608969.63,1057.24,10.7,9.83%,3.73%
2,At-Risk Loyal Customers,502,367.7,3.8,520919.39,1037.69,7.8,8.57%,3.19%
8,Regular Customers,782,202.4,2.2,510134.27,652.35,7.1,13.34%,3.12%
4,Low-Value Inactive Customers,1385,446.8,1.1,339291.29,244.98,4.0,23.63%,2.08%
7,New or Recent Customers,505,27.1,1.5,269807.83,534.27,8.0,8.62%,1.65%
0,At-Risk High-Spending Customers,69,424.5,1.7,165979.02,2405.49,7.3,1.18%,1.02%
6,Negative Net Value Customers,18,300.3,1.1,-1555.56,-86.42,4.3,0.31%,-0.01%


## RFM Segment Interpretation

The RFM segmentation shows a strong concentration of customer value. Champions represent around 21.84% of customers but contribute about 69.09% of total net product monetary value, making them the most important customer group for retention and loyalty strategies.

Potential Loyalists also show meaningful value contribution and may be good targets for engagement campaigns. At-risk customer groups have relatively high historical value but weaker recent activity, suggesting potential churn risk.

Low-value inactive customers represent a large share of the customer base but contribute limited monetary value. Negative net value customers are a very small group whose return or cancellation value exceeds their positive product purchase value.

RFM scores are relative to this dataset. A score of 5 means the customer is in a higher-ranked group within the current customer base, not that they passed a fixed business threshold.

In [11]:
rfm.to_csv("../data/processed/rfm_customer_segments.csv", index=False)
segment_summary.to_csv("../data/processed/rfm_segment_summary.csv", index=False)
segment_summary_formatted.to_csv("../data/processed/rfm_segment_summary_formatted.csv", index=False)

print("RFM customer segmentation outputs exported successfully.")

RFM customer segmentation outputs exported successfully.


In [12]:
negative_monetary_summary.to_csv("../data/processed/negative_monetary_summary.csv", index=False)